Schede concettuali prodotte da generative AI e poi passate al generatore di embedding

In [2]:
TESTO_DIR = "key_results_testi"


# ============================================================
# FUNZIONE: ESTRAI METADATA DAL NOME FILE
# ============================================================
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")

    # Cerca il pattern mese_anno_-_mese_anno
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)

    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')  # es. "Apr 2017"
        fine_periodo = periodo_match.group(2).replace('_', ' ')  # es. "Sep 2017"
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base

    # Paese: tutto quello che precede il primo mese
    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base

    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,  # es. "Apr 2017"
        "fine_periodo": fine_periodo,  # es. "Sep 2017"
        "nome_file": nome_file
    }

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. Configurazione del client Groq (Incolla qui la tua API Key di Groq)

client = Groq(api_key=GROQ_API_KEY)

# Usiamo Llama 3.3 70B: incredibilmente potente e preciso sui vincoli logici
MODELLO_LLAMA = "llama-3.3-70b-versatile"

OUTPUT_CSV = "schede_estratte_groq_llama3.csv"

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file da elaborare...")

# Prompt blindati per testo fluido senza etichette o rimasugli geografici
# PROMPT DI SISTEMA AGGIORNATO CON TAG SEMANTICI PER EMBEDDING
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain lines. Each line MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "The 4 required lines must start exactly with:\n"
    "- [DRIVERS AND ECONOMIC FACTORS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged lines. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Line 1 must start with [DRIVERS AND ECONOMIC FACTORS]: and focus on drivers, agriculture, and economic factors.\n"
    "2) Line 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Line 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Line 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods and nutrition.\n\n"
    "Report to analyze:\n"
)




print("Avvio estrazione tramite Llama via Groq API...")
risultati_finali = []

# Ciclo principale di elaborazione
for nome_file in tqdm(files[:5]):
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo = f.read()

    # Pulizia del testo secondo la tua regex nativa
    testo_pulito = re.sub(r'^.*?={40}\n\n', '', testo, flags=re.DOTALL).strip()
    if not testo_pulito:
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Gestione automatica dei limiti di token al minuto (TPM) del piano free di Groq
    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=500
            )

            # --- ESTRAZIONE STANDARD CORRETTA E SICURA ---
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            # Rilevamento dei limiti di velocità (Quota API esaurita al minuto)
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno di Sicurezza] Limite di Token raggiunto sul file {nome_file}. Attendo 65 secondi...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto sul file {nome_file}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Rimozione di sicurezza finale di eventuali anomalie grafiche (es. parentesi quadre)
        scheda_anonima = re.sub(r'\[.*?\]', '', scheda_anonima).strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Gestione metadati nativa
        metadata = estrai_metadata(nome_file) if 'estrai_metadata' in globals() else nome_file

        # Salvataggio incrementale nel CSV ad ogni iterazione
        nuovo_dato = pd.DataFrame([{
            "file": nome_file,
            "metadata": metadata,
            "testo_originale": testo_pulito,
            "scheda_llm": scheda_anonima
        }])

        nuovo_dato.to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV), encoding='utf-8')

        # Pausa preventiva obbligatoria (10 secondi) per distribuire i token nel piano gratuito di Groq
        time.sleep(10)
    else:
        print(f"\n[Salto File] Impossibile elaborare {nome_file} dopo 5 tentativi.")

print(f"\nProcesso completato! Il file finale pulito è salvato in: '{OUTPUT_CSV}'")

Trovati 497 file da elaborare...
Avvio estrazione tramite Llama via Groq API...


100%|██████████| 5/5 [00:55<00:00, 11.13s/it]


Processo completato! Il file finale pulito è salvato in: 'schede_estratte_groq_llama3.csv'


In [ ]:
df_groq4 = pd.read_csv('/content/schede_estratte_groq_llama3.csv')
df_groq4['scheda_llm'][1]

': The affected areas are expected to experience favourable agricultural performance and abundant rainfall, which is likely to improve household food stocks, despite the prices of manufactured products remaining higher than average due to the high cost of transportation.\n: In the current period, 1.9 million people are classified in IPC Phase 3 or above (Crisis or worse), which accounts for a significant portion of the population.\n: Nearly 1.2 million people (10 percent of the total population analysed) are projected to be in IPC Phase 3 (Crisis), indicating a marked improvement from the current period.\n: The humanitarian impacts on livelihoods and nutrition are significant, with a substantial number of people relying on limited food resources, and the situation is expected to affect the well-being of 1.2 million people, highlighting the need for continued support to address food insecurity.'

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. Configurazione (Ricordati di usare la NUOVA chiave revocando la vecchia)
client = Groq(api_key=GROQ_API_KEY)

MODELLO_LLAMA = "llama-3.3-70b-versatile"

# Percorsi dei file su Google Drive
CSV_INPUT_UNITI = "/content/drive/MyDrive/HERO/dataset_report_uniti.csv"
OUTPUT_CSV_ANONIMO = "/content/drive/MyDrive/HERO/dataset_report_anonimizzati.csv"

# Caricamento del dataset unificato
if os.path.exists(CSV_INPUT_UNITI):
    df_uniti = pd.read_csv(CSV_INPUT_UNITI)
    print(f"Caricato dataset unificato con {len(df_uniti)} righe.")
else:
    raise FileNotFoundError(f"Non ho trovato il file {CSV_INPUT_UNITI}.")

# --- PROMPT AGGIORNATI CON TUTTI I DRIVER E STRUTTURA FLUIDA ---
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

print("Avvio anonimizzazione tramite Llama via Groq API...")

# Ciclo principale sulle righe del DataFrame
for index, row in tqdm(df_uniti.iterrows(), total=len(df_uniti), desc="Elaborazione report"):

    # Controllo per riprendere il lavoro in caso di crash
    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if row['folder_name'] in df_check['folder_name'].values:
            continue

    testo_pulito = row['text']
    if pd.isna(testo_pulito) or not str(testo_pulito).strip():
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=800  # Spazio sufficiente per un testo unito e dettagliato
            )
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno di Sicurezza] Rate limit raggiunto. Attendo 65 secondi...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto alla riga {index}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Pulizia di sicurezza solo per rimasugli di markdown di intestazione (es. ###)
        # NON tocchiamo le parentesi quadre dei tag [SHOCKS AND DRIVERS] per non rovinare la struttura semantica del testo unito
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Creazione della riga con il testo unito pronto per l'embedder
        nuovo_dato = pd.DataFrame([{
            "folder_name": row['folder_name'],
            "country": row['country'],
            "start_period": row['start_period'],
            "end_period": row['end_period'],
            "language": row['language'],
            "testo_originale_unito": testo_pulito,
            "report_anonimo_unito": scheda_anonima  # <--- Questa è la colonna che manderai all'embedder
        }])

        # Salvataggio incrementale nel CSV ad ogni iterazione
        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

        # Pausa preventiva obbligatoria per il piano gratuito di Groq
        time.sleep(10)
    else:
        print(f"\n[Salto Riga] Impossibile elaborare l'indice {index} dopo 5 tentativi.")

print(f"\nProcesso completato! Il file pronto per l'embedder è: '{OUTPUT_CSV_ANONIMO}'")


In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 1. CONFIGURAZIONE CHIAVE API (Usa la nuova chiave generata nella console)
client = Groq(api_key=GROQ_API_KEY)

# Llama 3.3 70B: la scelta migliore per non perdere i dati numerici e le logiche di anonimizzazione
#MODELLO_LLAMA = "llama-3.3-70b-versatile"
MODELLO_LLAMA = "openai/gpt-oss-20b"

# Percorsi dei file su Google Drive
CSV_INPUT_UNITI = "/content/drive/MyDrive/HERO/dataset_report_uniti.csv"
OUTPUT_CSV_ANONIMO = "/content/drive/MyDrive/HERO/dataset_report_anonimizzati.csv"

# Assicuriamoci che il Drive sia montato prima di procedere
from google.colab import drive
drive.mount('/content/drive')

# Caricamento del dataset unificato
if os.path.exists(CSV_INPUT_UNITI):
    df_uniti = pd.read_csv(CSV_INPUT_UNITI)
    print(f"Caricato dataset unificato con {len(df_uniti)} righe. Pronto per Groq.")
else:
    raise FileNotFoundError(f"Non ho trovato il file {CSV_INPUT_UNITI}. Verifica i passaggi precedenti.")

# --- PROMPT OTTIMIZZATI CON TUTTI I DRIVER E STRUTTURA FLUIDA ---
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "CRITICAL RULE 5: Output Language. You MUST write the entire output in English, even if the input report is written in French, Spanish, or any other language.\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

print("\nAvvio anonimizzazione tramite Llama 3.3 70B via Groq API...")

# Ciclo principale sulle righe del DataFrame
for index, row in tqdm(df_uniti.iterrows(), total=len(df_uniti), desc="Elaborazione report"):

    # MECCANISMO DI PERSISTENZA: Se Colab si disconnette, salta le righe già fatte salvate nel CSV
    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if row['folder_name'] in df_check['folder_name'].values:
            continue

    testo_pulito = row['text']
    if pd.isna(testo_pulito) or not str(testo_pulito).strip():
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Ciclo di gestione degli errori e del rate limit (Fino a 5 tentativi per report)
    while not successo and tentativi < 5:
        try:
            completion = client.chat.completions.create(
                model=MODELLO_LLAMA,
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
                ],
                temperature=0.1,
                max_tokens=800  # Limite bilanciato per contenere i consumi del piano free
            )
            scheda_anonima = completion.choices[0].message.content.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            # Se colpiamo il limite di token al minuto (TPM), attiviamo il freno di emergenza di 65 secondi
            if "429" in errore_str or "rate_limit_exceeded" in errore_str or "RESOURCE_EXHAUSTED" in errore_str:
                print(f"\n[Freno Emergenza 429] Limite raggiunto alla riga {index}. Attendo 65 secondi per svuotare il contatore...")
                time.sleep(65)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto alla riga {index}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Pulizia di sicurezza solo per rimosso markdown di formattazione pesante (###)
        # NOTA: Manteniamo le parentesi quadre perché servono come ancore semantiche per l'embedder
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Creazione del nuovo record con il testo unito pronto per l'embedder
        nuovo_dato = pd.DataFrame([{
            "folder_name": row['folder_name'],
            "country": row['country'],
            "start_period": row['start_period'],
            "end_period": row['end_period'],
            "language": row['language'],
            "testo_originale_unito": testo_pulito,
            "report_anonimo_unito": scheda_anonima  # <-- Questa colonna andrà inviata al modello di embedding
        }])

        # Scrittura incrementale sul CSV su Drive (Massima sicurezza contro i crash di Colab)
        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

        # PAUSA STRATEGICA DI 40 SECONDI: Impedisce l'accumulo di token nel piano gratuito di Groq
        time.sleep(40)
    else:
        print(f"\n[Salto Riga] Impossibile elaborare l'indice {index} dopo 5 tentativi consecutivi.")

print(f"\nProcesso completato! Il file finale per gli embedding è salvato in: '{OUTPUT_CSV_ANONIMO}'")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Caricato dataset unificato con 492 righe. Pronto per Groq.

Avvio anonimizzazione tramite Llama 3.3 70B via Groq API...


Elaborazione report:   0%|          | 0/492 [00:00<?, ?it/s]


[Freno Emergenza 429] Limite raggiunto alla riga 1. Attendo 65 secondi per svuotare il contatore...

[Freno Emergenza 429] Limite raggiunto alla riga 1. Attendo 65 secondi per svuotare il contatore...


Elaborazione report:   0%|          | 1/492 [01:46<14:31:26, 106.49s/it]


KeyboardInterrupt: 

In [ ]:
import os
import re
import time
import pandas as pd
from tqdm import tqdm
from google.colab import userdata

# Installa la libreria ufficiale se mancante
try:
    import google.generativeai as genai
except ImportError:
    !pip install -q google-generativeai
    import google.generativeai as genai

# 1. CONFIGURAZIONE CHIAVE API GEMINI
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

# Usiamo Gemini 1.5 Flash: ideale per finestre di contesto enormi e piano gratuito generoso
model = genai.GenerativeModel('gemini-1.5-flash')

# Percorsi dei file su Google Drive
CSV_INPUT_UNITI = "/content/drive/MyDrive/HERO/dataset_report_uniti.csv"
OUTPUT_CSV_ANONIMO = "/content/drive/MyDrive/HERO/dataset_report_anonimizzati.csv"

# Assicuriamoci che il Drive sia montato
from google.colab import drive
drive.mount('/content/drive')

# Caricamento del dataset unificato
if os.path.exists(CSV_INPUT_UNITI):
    df_uniti = pd.read_csv(CSV_INPUT_UNITI)
    print(f"Caricato dataset unificato con {len(df_uniti)} righe. Pronto per Gemini.")
else:
    raise FileNotFoundError(f"Non ho trovato il file {CSV_INPUT_UNITI}.")

# --- PROMPT OTTIMIZZATI PER COMPRENDERE TUTTI I DRIVER ---
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "CRITICAL RULE 5: Output Language. You MUST write the entire output in English, even if the input report is written in French, Spanish, or any other language.\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

print("\nAvvio anonimizzazione tramite Gemini 1.5 Flash...")

# Ciclo principale sulle righe del DataFrame
for index, row in tqdm(df_uniti.iterrows(), total=len(df_uniti), desc="Elaborazione report"):

    # MECCANISMO DI PERSISTENZA: Salta le righe già elaborate nel CSV di output
    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if row['folder_name'] in df_check['folder_name'].values:
            continue

    testo_pulito = row['text']
    if pd.isna(testo_pulito) or not str(testo_pulito).strip():
        continue

    successo = False
    tentativi = 0
    scheda_anonima = ""

    # Uniamo i prompt e il testo da analizzare per Gemini
    prompt_completo = f"{prompt_sistema}\n\n{prompt_struttura}\n\nReport:\n{testo_pulito}"

    while not successo and tentativi < 5:
        try:
            # Configurazione della temperatura bassa per massima precisione logica
            response = model.generate_content(
                prompt_completo,
                generation_config={"temperature": 0.1}
            )
            scheda_anonima = response.text.strip()
            successo = True

        except Exception as e:
            errore_str = str(e)
            if "429" in errore_str or "Quota" in errore_str:
                print(f"\n[Rate Limit Gemini] Attendo 30 secondi prima di riprovare la riga {index}...")
                time.sleep(30)
                tentativi += 1
            else:
                print(f"\nErrore imprevisto alla riga {index}: {e}")
                tentativi += 1
                time.sleep(5)

    if successo:
        # Pulizia di sicurezza per rimosso markdown di formattazione pesante (###)
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Creazione del record
        nuovo_dato = pd.DataFrame([{
            "folder_name": row['folder_name'],
            "country": row['country'],
            "start_period": row['start_period'],
            "end_period": row['end_period'],
            "language": row['language'],
            "testo_originale_unito": testo_pulito,
            "report_anonimo_unito": scheda_anonima
        }])

        # Scrittura incrementale sul CSV su Drive
        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

        # Pausa minima precauzionale di 4 secondi (Gemini Free consente circa 15 richieste RPM)
        time.sleep(4)
    else:
        print(f"\n[Salto Riga] Impossibile elaborare l'indice {index} dopo 5 tentativi consecutivi.")

print(f"\nProcesso completato! Il file finale per gli embedding è salvato in: '{OUTPUT_CSV_ANONIMO}'")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Caricato dataset unificato con 492 righe. Pronto per Gemini.

Avvio anonimizzazione tramite Gemini 1.5 Flash...


Elaborazione report:   0%|          | 0/492 [00:00<?, ?it/s]WARNING:tornado.access:404 POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2517.58ms



Errore imprevisto alla riga 1: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.



Errore imprevisto alla riga 1: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


Elaborazione report:   0%|          | 1/492 [00:13<1:51:01, 13.57s/it]


KeyboardInterrupt: 

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
!pip install -q transformers accelerate bitsandbytes pandas tqdm huggingface_hub

In [5]:
import os
import re
import gc
import time
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# Forza il download accelerato anche per questa sessione Python
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# --- CONFIGURAZIONE MODELLO UFFICIALE META ---
NOME_MODELLO = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Configurazione BitsAndBytes per blindare il modello a 4-bit dentro la GPU T4
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Svuotiamo la cache prima di iniziare per recuperare ogni singolo MB di RAM liberi
torch.cuda.empty_cache()
gc.collect()

print("Scaricamento accelerato e caricamento del modello ufficiale Meta...")
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
model = AutoModelForCausalLM.from_pretrained(
    NOME_MODELLO,
    quantization_config=quantization_config,
    device_map="auto"
)

# Creazione della pipeline di generazione di testo
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Token di stop nativi di Llama 3.1 per bloccare immediatamente l'output a fine risposta
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

# --- CONFIGURAZIONE PERCORSI DRIVE ---
CSV_INPUT_UNITI = "/content/drive/MyDrive/HERO/dataset_report_uniti.csv"
OUTPUT_CSV_ANONIMO = "/content/drive/MyDrive/HERO/dataset_report_anonimizzati_locale.csv"

from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(CSV_INPUT_UNITI):
    df_uniti = pd.read_csv(CSV_INPUT_UNITI)
    print(f"\nCaricato dataset unificato con {len(df_uniti)} righe. Inizio elaborazione...")
else:
    raise FileNotFoundError(f"Impossibile trovare il file {CSV_INPUT_UNITI}.")

# --- PROMPT DI STRUTTURAZIONE ED ESTENSIONE DEI DRIVER ---
prompt_sistema = (
    "You are an expert humanitarian data analyst. Your job is to rewrite food insecurity reports into an anonymous summary optimized for vector embeddings.\n"
    "CRITICAL RULE 1: Absolute Anonymization. You must completely eliminate all names of countries, cities, regions, provinces, or dates (months/years).\n"
    "CRITICAL RULE 2: No Placeholders. Do NOT use tags like '[Country]' or '[Region]'. Use generic words like 'the affected areas', 'the population', or 'the country'.\n"
    "CRITICAL RULE 3: Exact Data Integrity. You MUST preserve all exact numbers, percentages, figures, and technical classifications (e.g., IPC Phase 3, 1.2 million, 10 percent). Do NOT convert numbers into generic words.\n"
    "CRITICAL RULE 4: Strict Section Tagging. You must structure your output using exactly 4 plain paragraphs. Each paragraph MUST start with the specific semantic tag provided, followed by a colon and the text. Do not use markdown headers (like ###) or bullet points (*).\n"
    "CRITICAL RULE 5: Output Language. You MUST write the entire output in English, even if the input report is written in French, Spanish, or any other language.\n"
    "The 4 required tags are:\n"
    "- [SHOCKS AND DRIVERS]: \n"
    "- [CURRENT FOOD SECURITY DATA]: \n"
    "- [PROJECTED FOOD SECURITY DATA]: \n"
    "- [HUMANITARIAN IMPACTS]: "
)

prompt_struttura = (
    "Analyze the following report and write a clear, anonymous summary structured into the 4 requested tagged sections. "
    "Maintain every single numerical figure, percentage, and IPC phase exactly as written, but anonymize all places and dates:\n"
    "1) Section 1 must start with [SHOCKS AND DRIVERS]: and focus on ALL underlying causes: economic factors, agriculture, climate shocks (rainfall, droughts), conflict, displacement, and disease outbreaks.\n"
    "2) Section 2 must start with [CURRENT FOOD SECURITY DATA]: and list exact technical indicators for the current period.\n"
    "3) Section 3 must start with [PROJECTED FOOD SECURITY DATA]: and list exact technical indicators for the projected period.\n"
    "4) Section 4 must start with [HUMANITARIAN IMPACTS]: and detail the humanitarian impacts on livelihoods, nutrition, and displacement consequences.\n\n"
    "Report to analyze:\n"
)

# Ciclo principale di scorrimento del DataFrame unificato
for index, row in tqdm(df_uniti.iterrows(), total=len(df_uniti), desc="Elaborazione report"):

    # Persistenza: salta se già fatto
    if os.path.exists(OUTPUT_CSV_ANONIMO):
        df_check = pd.read_csv(OUTPUT_CSV_ANONIMO)
        if row['folder_name'] in df_check['folder_name'].values:
            continue

    testo_pulito = row['text']
    if pd.isna(testo_pulito) or not str(testo_pulito).strip():
        continue

    # Formattazione del prompt
    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": f"{prompt_struttura}\n\nReport:\n{testo_pulito}"}
    ]

    prompt_formattato = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    try:
        # Svuotamento preventivo aggressivo della cache VRAM prima della chiamata
        torch.cuda.empty_cache()
        gc.collect()

        # Usiamo inference_mode per abbattere l'uso della RAM di calcolo ed evitare l'OOM
        with torch.inference_mode():
            outputs = pipe(
                prompt_formattato,
                max_new_tokens=700,      # Ottimizzato per contenere lo spazio di generazione
                do_sample=False,         # Massima precisione logica
                eos_token_id=terminators,
                pad_token_id=tokenizer.eos_token_id,
                return_full_text=False   # CRUCIALE: non alloca memoria per restituire il testo in ingresso
            )

        # Con return_full_text=False la stringa restituita contiene solo l'output dell'LLM
        scheda_anonima = outputs[0]["generated_text"].strip()
        scheda_anonima = re.sub(r'###.*?\n', '', scheda_anonima).strip()

        # Compilazione del record finale
        nuovo_dato = pd.DataFrame([{
            "folder_name": row['folder_name'],
            "country": row['country'],
            "start_period": row['start_period'],
            "end_period": row['end_period'],
            "language": row['language'],
            "testo_originale_unito": testo_pulito,
            "report_anonimo_unito": scheda_anonima
        }])

        # Scrittura sul CSV di sicurezza
        nuovo_dato.to_csv(OUTPUT_CSV_ANONIMO, mode='a', index=False, header=not os.path.exists(OUTPUT_CSV_ANONIMO), encoding='utf-8')

    except Exception as e:
        print(f"\nErrore riscontrato all'indice {index}: {e}")
        time.sleep(5)

print(f"\nProcesso completato! Il file finale pulito è in: '{OUTPUT_CSV_ANONIMO}'")



Scaricamento accelerato e caricamento del modello ufficiale Meta...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-6a58ce5d-5ebb25dc24571f777ad51456;9385f52e-aab9-42f7-a928-dcd7af3e45e8)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.